In [11]:
# imports
import os, unicodedata
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langchain_ollama import ChatOllama

In [12]:
# modelo local no Ollama (API nativa: sem /v1 no final da URL)
MODEL = 'qwen3:4b'
BASE_URL = 'http://localhost:11434'
modelo = ChatOllama(model=MODEL, base_url=BASE_URL,
    temperature=0, num_predict=256, validate_model_on_init=False)

print(f'Usando o modelo {MODEL} em {BASE_URL}')

Usando o modelo qwen3:4b em http://localhost:11434


In [13]:
# catálogo de setores da empresa de TI
SETORES = [
    {"nome_setor": "redes", "id": 1, "funcionario": "Ana"},
    {"nome_setor": "suporte tecnico", "id": 2, "funcionario": "Antonio"},
    {"nome_setor": "desenvolvimento", "id": 3, "funcionario": "Beatriz"}
]
    
def normalizar(texto: str) -> str:
    """Remove acentos, espaços e maiúsculas para comparar 'informatica' com 'Informática'."""
    sem_acento = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode()
    return sem_acento.strip().lower()

print(SETORES)

[{'nome_setor': 'redes', 'id': 1, 'funcionario': 'Ana'}, {'nome_setor': 'suporte tecnico', 'id': 2, 'funcionario': 'Antonio'}, {'nome_setor': 'desenvolvimento', 'id': 3, 'funcionario': 'Beatriz'}]


In [16]:
# agente coordenador de atendimento de chamados e direcionamento para o setor responsável (agente especializado)
Alessandro = create_agent(modelo, tools=[], system_prompt=(
    "Você é o agente de atendimento de chamados de uma empresa de TI. "
    "Você recebe o pedido do cliente e deve identificar o setor responsável pelo atendimento. "
    "Sempre chame a ferramenta criar_envelope passando o pedido do cliente e o setor responsável. "
    "Responda somente com o texto devolvido pela ferramenta, sem acrescentar nada.")
)

#permissões de acesso aos setores da empresa de TI
permissoes = {
    "redes": ["agente_rede"],
    "suporte_tecnico": ["agente_suporte_tecnico"],
    "desenvolvimento": ["agente_desenvolvimento"]
}

In [17]:
# função criar envelope de atendimento, que irá registrar a chamada ao setor responsável
def criar_envelope(conteudo, remetente, destinatario):
    return{
        "conteudo": conteudo,
        "remetente": remetente,
        "destinatario": destinatario
    }

In [ ]:
#ferramenta de comunicação entre agentes

#ferramenta inicial para encaminhar o envelope de atendimento para o setor responsável conforme a lista de permissões de acesso aos setores da empresa de TI
@tool
def enviar_para_supervisor(conteudo) -> str:
    """
    Envia o envelope de atendimento para o dicionário e busca o supervisor, que irá direcionar para o setor responsável.
    """
    envelope = criar_envelope(conteudo, remetente="agente_chamados", destinatario="supervisor")

    #verificando se o destinatário do envelope está na lista de permissões
    for envelope["destinatario"] in permissoes:
        permissoes[envelope["destinatario"]]

        return f"Envelope enviado para o supervisor: {envelope}"

    return f"Destinatário {envelope['destinatario']} não encontrado na lista de permissões."

# ferramenta de setor de rede - repassar o envelope de atendimento para o setor de redes
@tool
def setor_redes(conteudo, remetente):
    """Cria um envelope de atendimento para o setor de redes, com o conteúdo descrito pelo cliente e o remetente."""
    
    return criar_envelope(conteudo, remetente, "redes")

# ferramenta de setor de suporte técnico - repassar o envelope de atendimento para o setor de suporte técnico
@tool
def setor_suporte_tecnico(conteudo, remetente):
    """Cria um envelope de atendimento para o setor de suporte técnico, com o conteúdo descrito pelo cliente e o remetente."""
    
    return criar_envelope(conteudo, remetente, "suporte_tecnico")

# ferramenta de setor de desenvolvimento - repassar o envelope de atendimento para o setor de desenvolvimento
@tool
def setor_desenvolvimento(conteudo, remetente):
    """Cria um envelope de atendimento para o setor de desenvolvimento, com o conteúdo descrito pelo cliente e o remetente."""
    
    return criar_envelope(conteudo, remetente, "desenvolvimento")


In [ ]:
# demais agentes dos setores especializados (redes, suporte técnico e desenvolvimento) podem ser criados de forma similar, cada um com suas ferramentas específicas para processar os envelopes recebidos.
agente_rede = create_agent(modelo, tools=[setor_redes], system_prompt=(
    "Você é o agente do setor de redes de uma empresa de TI. "
    "Você recebe o envelope de atendimento do agente de chamados e deve processar o problema do cliente. "
    "Responda somente com o texto devolvido pela ferramenta, sem acrescentar nada.")
)
agente_suporte_tecnico = create_agent(modelo, tools=[setor_suporte_tecnico], system_prompt=(
    "Você é o agente do setor de suporte técnico de uma empresa de TI. "
    "Você recebe o envelope de atendimento do agente de chamados e deve processar o problema do cliente. "
    "Responda somente com o texto devolvido pela ferramenta, sem acrescentar nada.")
)

agente_desenvolvimento = create_agent(modelo, tools=[setor_desenvolvimento], system_prompt=(
    "Você é o agente do setor de desenvolvimento de uma empresa de TI. "
    "Você recebe o envelope de atendimento do agente de chamados e deve processar o problema do cliente. "
    "Responda somente com o texto devolvido pela ferramenta, sem acrescentar nada.")
)


In [ ]:
# chamando os agentes para processar um chamado de atendimento do cliente


In [ ]:
# entrada do usuário (cliente) para o agente de atendimento de chamados
pedido_cliente = input()